# 01 — Tạo dataset hoàn chỉnh cho EFormExcelMapper

Notebook này dùng cho đúng bài toán:

> **Excel Header + Context → DataField eForm**

Không train model bằng value của từng ô.

## Luồng dữ liệu

```text
../Data/Raw/*.xlsx + ../Data/Labels/eform_fields.jsonl
        ↓
   Parse Excel
        ↓
Dựng Header Context
        ↓
tool/work/Labels/mappings.jsonl
        ↓
Bạn xác nhận DataField đúng
        ↓
verified = true
        ↓
tool/artifacts/dataset/Train/train.jsonl
tool/artifacts/dataset/Validation/validation.jsonl
tool/artifacts/dataset/Test/test.jsonl
tool/artifacts/dataset/Test/ground_truth.jsonl
```

## Cấu trúc thư mục

```text
Module/AI Import/
├── Data/                         # chỉ đọc dữ liệu nguồn
│   ├── Raw/
│   └── Labels/eform_fields.jsonl
└── tool/
    ├── work/Labels/mappings.jsonl # file cần người duyệt
    └── artifacts/dataset/         # đầu ra cho notebook 02
```

### Nguyên tắc

- `../Data/Raw` giữ nguyên các file `.xlsx`; notebook không sửa file Excel.
- `tool/work/Labels/mappings.jsonl` là nơi duyệt ground truth.
- Không tự động coi mapping candidate là ground truth.
- Train/Validation/Test được chia theo **workbook**, tránh data leakage.
- Một sample train tương ứng với **một mapping cột Excel → DataField**.

In [ ]:
# CELL 1 - Cài thư viện

%pip install -q openpyxl rapidfuzz tqdm pandas

%pip install  pandas
%pip install pandas openpyxl rapidfuzz

In [ ]:
# CELL 2 - Import

from __future__ import annotations

import json
import re
import hashlib
import random
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from rapidfuzz import fuzz
from tqdm.auto import tqdm

print("OK")

In [ ]:
# CELL 3 - CONFIG

RUN_ON_COLAB = False
MODULE_ROOT_OVERRIDE = ""  # Chỉ điền khi notebook không nằm trong repository.

if RUN_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MODULE_ROOT = Path("/content/drive/MyDrive/eform_btp/Module/AI Import")
elif MODULE_ROOT_OVERRIDE:
    MODULE_ROOT = Path(MODULE_ROOT_OVERRIDE).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    MODULE_ROOT = None
    for base in candidates:
        if (base / "Data").is_dir() and (base / "Main").is_dir():
            MODULE_ROOT = base
            break
        nested = base / "Module" / "AI Import"
        if (nested / "Data").is_dir() and (nested / "Main").is_dir():
            MODULE_ROOT = nested
            break
    if MODULE_ROOT is None:
        raise FileNotFoundError("Không tìm thấy Module/AI Import. Hãy đặt MODULE_ROOT_OVERRIDE.")

import shutil
TOOL_ROOT = MODULE_ROOT / "tool"
SOURCE_DATA_ROOT = MODULE_ROOT / "Data"
WORK_ROOT = TOOL_ROOT / "work"
OUTPUT_ROOT = TOOL_ROOT / "artifacts" / "dataset"
RAW_DIR = SOURCE_DATA_ROOT / "Raw"
LABEL_DIR = WORK_ROOT / "Labels"
TRAIN_DIR = OUTPUT_ROOT / "Train"
VAL_DIR = OUTPUT_ROOT / "Validation"
TEST_DIR = OUTPUT_ROOT / "Test"

MANIFEST_PATH = WORK_ROOT / "Manifest.jsonl"
MAPPING_PATH = LABEL_DIR / "mappings.jsonl"
SOURCE_MAPPING_PATH = SOURCE_DATA_ROOT / "Labels" / "mappings.jsonl"

# Metadata DataField lấy từ eForm/database.
EFORM_FIELDS_PATH = SOURCE_DATA_ROOT / "Labels" / "eform_fields.jsonl"
if not EFORM_FIELDS_PATH.exists():
    EFORM_FIELDS_PATH = SOURCE_DATA_ROOT / "eform_fields.jsonl"

TRAIN_PATH = TRAIN_DIR / "train.jsonl"
VAL_PATH = VAL_DIR / "validation.jsonl"
TEST_PATH = TEST_DIR / "test.jsonl"
GROUND_TRUTH_PATH = TEST_DIR / "ground_truth.jsonl"

for folder in [
    LABEL_DIR,
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Thiếu thư mục Excel nguồn: {RAW_DIR}")
if not MAPPING_PATH.exists() and SOURCE_MAPPING_PATH.exists():
    shutil.copy2(SOURCE_MAPPING_PATH, MAPPING_PATH)
    print("Đã tạo bản nhãn làm việc; dữ liệu nguồn không bị sửa.")

SEED = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

MAX_HEADER_SCAN_ROWS = 80
MAX_SAMPLE_VALUES = 8
HARD_NEGATIVE_COUNT = 4

print("MODULE_ROOT:", MODULE_ROOT)
print("RAW_DIR:", RAW_DIR)
print("MAPPING_PATH:", MAPPING_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
# CELL 4 - Helpers

def normalize_text(value: Any) -> str:
    if value is None:
        return ""

    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s*\n\s*", " ", text)

    return text.strip()


def norm_key(value: Any) -> str:
    return re.sub(
        r"\s+",
        " ",
        normalize_text(value).lower()
    )


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []

    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as ex:
                raise ValueError(
                    f"Lỗi JSON tại {path}:{line_number}: {ex}"
                )

    return rows


def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                )
                + "\n"
            )


def stable_id(*parts: Any) -> str:
    raw = "||".join(
        normalize_text(x)
        for x in parts
    )

    return hashlib.sha1(
        raw.encode("utf-8")
    ).hexdigest()[:16]


def is_empty_value(value: Any) -> bool:
    return normalize_text(value) in {
        "",
        "-",
        "–",
        "—",
        "...",
        "…"
    }


print("Helpers OK")

# Phần 1 — Đọc cấu trúc Excel

Notebook ưu tiên các biểu có dòng mã cột:

```text
A | B | (1) | (2) | (3) | ...
```

Đây là kiểu rất phổ biến trong eForm.

Ví dụ:

```text
Số VBQPPL đã được ban hành tại cấp xã
    └── Chia theo tên loại VBQPPL
        └── Quyết định của UBND
            └── (3)
```

Parser sẽ dựng thành `header_path`:

```json
[
  "Số VBQPPL đã được ban hành tại cấp xã",
  "Chia theo tên loại VBQPPL",
  "Quyết định của UBND"
]
```

In [ ]:
# CELL 5 - Excel structure parser

NUMERIC_CODE_RE = re.compile(
    r"^\(\s*(\d+)\s*\)$"
)

ANY_COLUMN_CODE_RE = re.compile(
    r"^(?:[A-Z]{1,3}|\(\s*\d+\s*\))$"
)

DOC_TYPE_PATTERNS = [
    re.compile(
        r"Biểu\s*số\s*:\s*([^\s\n]+)",
        re.IGNORECASE
    ),
    re.compile(
        r"\b(\d+[a-zA-Z]?/[A-ZÀ-Ỹ0-9\-Đ]+(?:/[A-ZÀ-Ỹ0-9\-Đ]+)*)\b"
    )
]


def is_numeric_code(value: Any) -> bool:
    return bool(
        NUMERIC_CODE_RE.fullmatch(
            normalize_text(value)
        )
    )


def is_any_column_code(value: Any) -> bool:
    return bool(
        ANY_COLUMN_CODE_RE.fullmatch(
            normalize_text(value)
        )
    )


def extract_doc_type_from_text(
    text: str
) -> Optional[str]:

    text = normalize_text(text)

    for pattern in DOC_TYPE_PATTERNS:
        match = pattern.search(text)

        if match:
            return match.group(1).strip()

    return None


def build_merged_lookup(ws):
    """
    Với merged C3:E3:
    mọi cell C3,D3,E3 đều đọc value của C3.
    """
    lookup = {}

    for merged_range in ws.merged_cells.ranges:
        min_col, min_row, max_col, max_row = \
            merged_range.bounds

        for row in range(
            min_row,
            max_row + 1
        ):
            for col in range(
                min_col,
                max_col + 1
            ):
                lookup[(row, col)] = (
                    min_row,
                    min_col
                )

    return lookup


def effective_value(
    ws,
    row: int,
    col: int,
    merged_lookup
):
    source = merged_lookup.get(
        (row, col)
    )

    if source:
        return ws.cell(
            source[0],
            source[1]
        ).value

    return ws.cell(
        row,
        col
    ).value


def find_column_code_row(ws) -> Optional[int]:
    """
    Tìm dòng chứa:
    A, B, (1), (2), (3), ...
    """

    best_row = None
    best_score = -1

    max_row = min(
        ws.max_row,
        MAX_HEADER_SCAN_ROWS
    )

    for row in range(
        1,
        max_row + 1
    ):
        values = [
            ws.cell(row, col).value
            for col in range(
                1,
                ws.max_column + 1
            )
        ]

        non_empty = [
            value
            for value in values
            if normalize_text(value)
        ]

        if not non_empty:
            continue

        numeric_codes = sum(
            is_numeric_code(value)
            for value in non_empty
        )

        all_codes = sum(
            is_any_column_code(value)
            for value in non_empty
        )

        # Ít nhất 2 mã (n) để tránh nhầm STT dữ liệu.
        if numeric_codes < 2:
            continue

        score = (
            numeric_codes * 10
            + all_codes
        )

        if score > best_score:
            best_score = score
            best_row = row

    return best_row


def detect_doc_type(
    ws,
    header_end_row: int,
    merged_lookup
) -> str:

    seen = set()

    for row in range(
        1,
        min(header_end_row, 20) + 1
    ):
        for col in range(
            1,
            ws.max_column + 1
        ):
            source = merged_lookup.get(
                (row, col),
                (row, col)
            )

            if source in seen:
                continue

            seen.add(source)

            text = normalize_text(
                ws.cell(
                    source[0],
                    source[1]
                ).value
            )

            result = extract_doc_type_from_text(
                text
            )

            if result:
                return result

    return ""


def detect_report_title(
    ws,
    header_end_row: int,
    merged_lookup
) -> str:

    candidates = []
    seen = set()

    for row in range(
        1,
        min(header_end_row, 15) + 1
    ):
        for col in range(
            1,
            ws.max_column + 1
        ):
            source = merged_lookup.get(
                (row, col),
                (row, col)
            )

            if source in seen:
                continue

            seen.add(source)

            cell = ws.cell(
                source[0],
                source[1]
            )

            text = normalize_text(
                cell.value
            )

            if len(text) < 10:
                continue

            score = len(text)

            try:
                if cell.font.bold:
                    score += 20
            except Exception:
                pass

            candidates.append(
                (score, text)
            )

    if not candidates:
        return ""

    candidates.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    return candidates[0][1]


def build_header_path(
    ws,
    column: int,
    column_code_row: int,
    merged_lookup
) -> List[str]:

    path = []

    # Chỉ lấy block header gần dòng mã cột; không kéo dữ liệu của bảng trước
    # (ví dụ 77, ghi chú) vào prompt của bảng sau.
    header_start_row = max(1, column_code_row - 8)
    for candidate_row in range(column_code_row - 1, header_start_row - 1, -1):
        if not any(normalize_text(ws.cell(candidate_row, c).value) for c in range(1, ws.max_column + 1)):
            header_start_row = candidate_row + 1
            break

    for row in range(
        header_start_row,
        column_code_row
    ):
        value = effective_value(
            ws,
            row,
            column,
            merged_lookup
        )

        text = normalize_text(value)

        if not text:
            continue

        if is_any_column_code(text):
            continue

        # Loại duplicate do merged-cell propagate.
        if (
            path
            and norm_key(path[-1])
            == norm_key(text)
        ):
            continue

        path.append(text)

    return path


def get_sample_values(
    ws,
    column: int,
    data_start_row: int
) -> List[Any]:

    result = []

    end_row = min(
        ws.max_row,
        data_start_row + 200
    )

    for row in range(
        data_start_row,
        end_row + 1
    ):
        value = ws.cell(
            row,
            column
        ).value

        if is_empty_value(value):
            continue

        if isinstance(
            value,
            (str, int, float, bool)
        ):
            safe_value = value
        else:
            safe_value = str(value)

        if safe_value not in result:
            result.append(safe_value)

        if (
            len(result)
            >= MAX_SAMPLE_VALUES
        ):
            break

    return result


def infer_data_type(
    values: List[Any]
) -> str:

    if not values:
        return "unknown"

    numeric = sum(
        isinstance(
            value,
            (int, float)
        )
        and not isinstance(value, bool)
        for value in values
    )

    text = sum(
        isinstance(value, str)
        for value in values
    )

    total = len(values)

    if numeric / total >= 0.8:
        return "number"

    if text / total >= 0.8:
        return "text"

    return "mixed"

In [ ]:
# CELL 6 - Extract một workbook

def extract_workbook(
    excel_path: Path
) -> List[Dict[str, Any]]:

    wb = load_workbook(
        excel_path,
        data_only=False,
        read_only=False
    )

    records = []

    for ws in wb.worksheets:

        if (
            ws.max_row <= 1
            or ws.max_column <= 1
        ):
            continue

        merged_lookup = \
            build_merged_lookup(ws)

        code_row = \
            find_column_code_row(ws)

        if code_row is None:
            # Không cố đoán mapping nếu cấu trúc quá lạ.
            # Giữ nguyên để sau này có thể bổ sung parser.
            continue

        doc_type_code = detect_doc_type(
            ws,
            code_row,
            merged_lookup
        )

        report_title = detect_report_title(
            ws,
            code_row,
            merged_lookup
        )

        data_start_row = code_row + 1

        for column in range(
            1,
            ws.max_column + 1
        ):
            column_code = normalize_text(
                ws.cell(
                    code_row,
                    column
                ).value
            )

            # Dataset mapping chỉ lấy các cột (1),(2),(3)...
            if not is_numeric_code(
                column_code
            ):
                continue

            header_path = build_header_path(
                ws,
                column,
                code_row,
                merged_lookup
            )

            sample_values = get_sample_values(
                ws,
                column,
                data_start_row
            )

            record_id = stable_id(
                excel_path.name,
                ws.title,
                get_column_letter(column),
                column_code,
                "|".join(header_path)
            )

            records.append({
                "sample_id": record_id,

                "file": excel_path.name,
                "sheet": ws.title,

                "doc_type_code":
                    doc_type_code,

                "report_title":
                    report_title,

                "excel_column":
                    get_column_letter(column),

                "column_code":
                    column_code,

                "header_path":
                    header_path,

                "header":
                    header_path[-1]
                    if header_path
                    else "",

                "data_type":
                    infer_data_type(
                        sample_values
                    ),

                "sample_values":
                    sample_values,

                # ------- GROUND TRUTH -------
                "target_data_field_id": "",
                "target_field_name": "",
                "target_field_title": "",
                "target_column_code": "",

                "verified": False,
                "ignore": False,

                # Chỉ hỗ trợ review.
                "candidate_fields": []
            })

    wb.close()

    return records

# Phần 2 — Quét toàn bộ Raw

Bạn chỉ cần ném các Excel vào:

```text
Data/Raw/
```

Không cần chia theo biểu.

`Manifest.jsonl` chỉ dùng để:

- biết file nào đang enable;
- override `DocType` nếu parser không đọc được;
- cố định `split` train/validation/test nếu muốn.

In [ ]:
# CELL 7 - Scan Raw + Manifest

extensions = {
    ".xlsx",
    ".xlsm"
}

raw_files = sorted([
    file
    for file in RAW_DIR.rglob("*")
    if (
        file.is_file()
        and file.suffix.lower()
        in extensions
        and not file.name.startswith("~$")
    )
])

old_manifest = {
    row["file"]: row
    for row in read_jsonl(
        MANIFEST_PATH
    )
    if row.get("file")
}

manifest = []

for file in raw_files:
    old = old_manifest.get(
        file.name,
        {}
    )

    manifest.append({
        "file":
            file.name,

        "relative_path":
            str(
                file.relative_to(
                    SOURCE_DATA_ROOT
                )
            ).replace("\\", "/"),

        "doc_type_code":
            old.get(
                "doc_type_code",
                ""
            ),

        # train / validation / test
        # hoặc để trống để tự chia.
        "split":
            old.get(
                "split",
                ""
            ),

        "enabled":
            old.get(
                "enabled",
                True
            )
    })

write_jsonl(
    MANIFEST_PATH,
    manifest
)

print(
    "Số workbook:",
    len(raw_files)
)

pd.DataFrame(
    manifest
).head(20)

In [ ]:
# CELL 8 - Parse Raw → mappings.jsonl
#
# Nếu mappings.jsonl đã tồn tại:
# giữ nguyên label cũ theo sample_id.

old_labels = {
    row["sample_id"]: row
    for row in read_jsonl(
        MAPPING_PATH
    )
    if row.get("sample_id")
}

manifest_by_file = {
    row["file"]: row
    for row in manifest
}

records = []
errors = []

for file in tqdm(
    raw_files,
    desc="Parsing Excel"
):
    config = manifest_by_file.get(
        file.name,
        {}
    )

    if not config.get(
        "enabled",
        True
    ):
        continue

    try:
        extracted = \
            extract_workbook(file)

        override_doc_type = \
            normalize_text(
                config.get(
                    "doc_type_code",
                    ""
                )
            )

        for row in extracted:

            if override_doc_type:
                row["doc_type_code"] = \
                    override_doc_type

            old = old_labels.get(
                row["sample_id"]
            )

            if old:
                # Giữ các trường do người dùng xác nhận.
                for key in [
                    "target_data_field_id",
                    "target_field_name",
                    "target_field_title",
                    "target_column_code",
                    "verified",
                    "ignore",
                    "candidate_fields"
                ]:
                    if key in old:
                        row[key] = old[key]

            records.append(row)

    except Exception as ex:
        errors.append({
            "file": file.name,
            "error": repr(ex)
        })

write_jsonl(
    MAPPING_PATH,
    records
)

print(
    "Mapping candidates:",
    len(records)
)

print(
    "Workbook lỗi:",
    len(errors)
)

if errors:
    pd.DataFrame(errors)
else:
    print("Không có lỗi.")

In [ ]:
# CELL 9 - Xem mapping candidate

preview_rows = []

for row in records[:50]:
    preview_rows.append({
        "sample_id":
            row["sample_id"],

        "file":
            row["file"],

        "sheet":
            row["sheet"],

        "doc_type":
            row["doc_type_code"],

        "column":
            row["excel_column"],

        "code":
            row["column_code"],

        "header_path":
            " > ".join(
                row["header_path"]
            ),

        "datatype":
            row["data_type"],

        "target_field":
            row["target_data_field_id"],

        "verified":
            row["verified"]
    })

pd.DataFrame(
    preview_rows
)

# Phần 3 — Xác nhận Ground Truth

Sau Cell 8, mở:

```text
tool/work/Labels/mappings.jsonl
```

Ví dụ một record parser sinh ra:

```json
{
  "doc_type_code": "01c/TP/BH-TĐG",
  "header_path": [
    "Số VBQPPL đã được ban hành tại cấp xã",
    "Chia theo tên loại VBQPPL",
    "Quyết định của UBND"
  ],
  "column_code": "(3)"
}
```

Bạn xác nhận:

```json
{
  "target_data_field_id": "10503",
  "target_field_name": "a3",
  "target_field_title": "Số VBQPPL đã được ban hành tại cấp xã - Quyết định của UBND",
  "target_column_code": "(3)",
  "verified": true
}
```

Nếu cột không cần import:

```json
"ignore": true
```

> Không được `verified=true` nếu chưa chắc chắn.

In [ ]:
# CELL 10 - OPTIONAL
# Candidate suggestion từ eform_fields.jsonl
#
# Candidate chỉ giúp gán label nhanh hơn.
# Không tự verified.

def source_candidate_text(
    row: Dict[str, Any]
) -> str:

    parts = [
        row.get(
            "report_title",
            ""
        ),
        *row.get(
            "header_path",
            []
        ),
        row.get(
            "column_code",
            ""
        )
    ]

    return " | ".join([
        normalize_text(x)
        for x in parts
        if normalize_text(x)
    ])


def target_candidate_text(
    field: Dict[str, Any]
) -> str:

    return " | ".join([
        normalize_text(x)
        for x in [
            field.get(
                "field_title",
                ""
            ),
            field.get(
                "field_name",
                ""
            ),
            field.get(
                "column_code",
                ""
            )
        ]
        if normalize_text(x)
    ])


if EFORM_FIELDS_PATH.exists():

    fields = read_jsonl(
        EFORM_FIELDS_PATH
    )

    for row in records:

        if (
            row.get("verified")
            or row.get("ignore")
        ):
            continue

        same_doc_type = [
            field\
            for field in fields
            if (
                row["doc_type_code"]
                and norm_key(
                    field.get(
                        "doc_type_code"
                    )
                )
                ==
                norm_key(
                    row["doc_type_code"]
                )
            )
        ]

        pool = (
            same_doc_type
            if same_doc_type
            else fields
        )

        scored = []

        source_text = \
            source_candidate_text(row)

        for field in pool:

            score = float(
                fuzz.token_set_ratio(
                    source_text,
                    target_candidate_text(
                        field
                    )
                )
            )

            if (
                normalize_text(
                    row["column_code"]
                )
                ==
                normalize_text(
                    field.get(
                        "column_code"
                    )
                )
            ):
                score += 20

            scored.append({
                "data_field_id":
                    field.get(
                        "data_field_id",
                        ""
                    ),

                "field_name":
                    field.get(
                        "field_name",
                        ""
                    ),

                "field_title":
                    field.get(
                        "field_title",
                        ""
                    ),

                "column_code":
                    field.get(
                        "column_code",
                        ""
                    ),

                "score":
                    round(
                        min(
                            score,
                            100
                        ),
                        2
                    )
            })

        row["candidate_fields"] = \
            sorted(
                scored,
                key=lambda x:
                    x["score"],
                reverse=True
            )[:5]

    write_jsonl(
        MAPPING_PATH,
        records
    )

    print(
        "Đã cập nhật candidate_fields."
    )

else:
    print(
        "Không có eform_fields.jsonl."
    )

In [ ]:
# CELL 10A - Gợi ý theo template; TUYỆT ĐỐI không tự gắn verified
TEMPLATE_MAP_PATH = SOURCE_DATA_ROOT / "template_field_map.jsonl"
template_rows = []
if TEMPLATE_MAP_PATH.exists():
    template_rows = [json.loads(line) for line in TEMPLATE_MAP_PATH.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
template_by_field = {}
for item in template_rows:
    key = normalize_text(item.get("field_name"))
    if key: template_by_field.setdefault(key, []).append(item)
def short_doctype(value):
    m = re.search(r'(?<!\d)(\d{2}[a-z]?)(?![a-z])', normalize_text(value), re.I)
    return m.group(1).lower() if m else ''
approved_methods = {'human', 'curated', 'reviewed'}
suggested_count = 0
for row in records:
    # Nhãn legacy từng được auto-verify phải quay lại hàng chờ duyệt.
    if row.get('verified') and row.get('verification_method') not in approved_methods:
        row['verified'] = False
        row['needs_review'] = True
    if row.get('verified') or row.get('ignore'): continue
    source_doc = short_doctype(row.get('doc_type_code'))
    candidates = [c for c in row.get('candidate_fields', []) if short_doctype(c.get('doc_type_code', source_doc)) == source_doc and normalize_text(c.get('column_code')) == normalize_text(row.get('column_code'))]
    exact = []
    for c in candidates:
        for item in template_by_field.get(normalize_text(c.get('field_name')), []):
            if short_doctype(item.get('doc_type_code')) == source_doc and item.get('data_field_id'):
                exact.append((c, item))
    suggestion = exact[0][0] if len(exact) == 1 else (candidates[0] if len(candidates) == 1 and candidates[0].get('score', 0) >= 90 else None)
    if suggestion:
        row['suggested_data_field_id'] = suggestion.get('data_field_id', '')
        row['suggested_field_name'] = suggestion.get('field_name', '')
        row['suggested_field_title'] = suggestion.get('field_title', '')
        row['suggestion_method'] = 'template_or_rule'
        row['needs_review'] = True
        suggested_count += 1
write_jsonl(MAPPING_PATH, records)
print('Suggestions cần người duyệt:', suggested_count, '/', len(records))

# Phần 4 — Build dataset cho BGE-M3

Chỉ những record:

```json
"verified": true
```

mới được đưa vào dataset.

### Query

```text
[TASK] Excel to eForm DataField Mapping
[DOCTYPE] 01c/TP/BH-TĐG
[REPORT] ...
[PARENT] ...
[HEADER] Quyết định của UBND
[CODE] (3)
[TYPE] number
```

### Positive

```text
[DOCTYPE] 01c/TP/BH-TĐG
[FIELD] Số VBQPPL đã được ban hành tại cấp xã - Quyết định của UBND
[CODE] (3)
```

**Không đưa `DataFieldId` vào text embedding** để tránh model học thuộc ID vô nghĩa.  
`DataFieldId` chỉ giữ dưới dạng metadata/ground truth.

In [ ]:
# CELL 11 - Load + validate Verified Labels

labels = read_jsonl(
    MAPPING_PATH
)

verified = []
pending = []
ignored = []
invalid = []

for row in labels:

    if row.get("ignore"):
        ignored.append(row)
        continue

    if not row.get("verified"):
        pending.append(row)
        continue

    if row.get("verification_method") not in approved_methods:
        invalid.append(row)
        continue

    required = [
        row.get(
            "target_data_field_id"
        ),
        row.get(
            "target_field_title"
        )
    ]

    if not all(
        normalize_text(x)
        for x in required
    ):
        invalid.append(row)
        continue

    verified.append(row)

print(
    "Verified:",
    len(verified)
)

print(
    "Pending:",
    len(pending)
)

print(
    "Ignored:",
    len(ignored)
)

print(
    "Invalid:",
    len(invalid)
)

assert not invalid, (
    "Có record verified=true nhưng "
    "thiếu DataFieldId hoặc field_title."
)

In [ ]:
# CELL 12 - Build text representation

def build_query(
    row: Dict[str, Any]
) -> str:

    lines = [
        "[TASK] Excel to eForm DataField Mapping"
    ]

    doc_type = normalize_text(
        row.get(
            "doc_type_code"
        )
    )

    if doc_type:
        lines.append(
            f"[DOCTYPE] {doc_type}"
        )

    report_title = normalize_text(
        row.get(
            "report_title"
        )
    )

    if report_title:
        lines.append(
            f"[REPORT] {report_title}"
        )

    header_path = [
        normalize_text(x)
        for x in row.get(
            "header_path",
            []
        )
        if normalize_text(x)
    ]

    if len(header_path) > 1:
        lines.append(
            "[PARENT] "
            + " > ".join(
                header_path[:-1]
            )
        )

    if header_path:
        lines.append(
            f"[HEADER] {header_path[-1]}"
        )

    column_code = normalize_text(
        row.get(
            "column_code"
        )
    )

    if column_code:
        lines.append(
            f"[CODE] {column_code}"
        )

    datatype = normalize_text(
        row.get(
            "data_type"
        )
    )

    if datatype:
        lines.append(
            f"[TYPE] {datatype}"
        )

    return "\n".join(lines)


def build_target_text(
    row: Dict[str, Any]
) -> str:

    lines = []

    doc_type = normalize_text(
        row.get(
            "doc_type_code"
        )
    )

    if doc_type:
        lines.append(
            f"[DOCTYPE] {doc_type}"
        )

    field_name = normalize_text(
        row.get(
            "target_field_name"
        )
    )

    if field_name:
        lines.append(
            f"[FIELD_NAME] {field_name}"
        )

    field_title = normalize_text(
        row.get(
            "target_field_title"
        )
    )

    lines.append(
        f"[FIELD] {field_title}"
    )

    column_code = (
        normalize_text(
            row.get(
                "target_column_code"
            )
        )
        or
        normalize_text(
            row.get(
                "column_code"
            )
        )
    )

    if column_code:
        lines.append(
            f"[CODE] {column_code}"
        )

    return "\n".join(lines)


if verified:
    print(
        build_query(
            verified[0]
        )
    )

    print("\n--- POSITIVE ---\n")

    print(
        build_target_text(
            verified[0]
        )
    )

In [ ]:
# CELL 13 - Hard Negative Mining

def build_field_pool(
    rows: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:

    unique = {}

    for row in rows:

        key = (
            norm_key(
                row.get(
                    "doc_type_code"
                )
            ),
            str(
                row.get(
                    "target_data_field_id"
                )
            )
        )

        if key not in unique:
            unique[key] = row

    return list(
        unique.values()
    )


FIELD_POOL = build_field_pool(
    verified
)


def mine_negatives(
    row: Dict[str, Any],
    count: int = HARD_NEGATIVE_COUNT
) -> List[str]:

    correct_id = str(
        row["target_data_field_id"]
    )

    doc_type = norm_key(
        row.get(
            "doc_type_code"
        )
    )

    query = build_query(row)

    # Ưu tiên DataField cùng biểu.
    pool = [
        candidate
        for candidate in FIELD_POOL
        if (
            str(
                candidate[
                    "target_data_field_id"
                ]
            )
            != correct_id
            and
            norm_key(
                candidate.get(
                    "doc_type_code"
                )
            )
            == doc_type
        )
    ]

    # Nếu cùng biểu chưa đủ field:
    if len(pool) < count:
        extras = [
            candidate
            for candidate in FIELD_POOL
            if (
                str(
                    candidate[
                        "target_data_field_id"
                    ]
                )
                != correct_id
                and candidate not in pool
            )
        ]

        pool.extend(extras)

    scored = []

    for candidate in pool:

        target_text = \
            build_target_text(
                candidate
            )

        score = float(
            fuzz.token_set_ratio(
                query,
                target_text
            )
        )

        scored.append(
            (
                score,
                target_text
            )
        )

    scored.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    negatives = []
    seen = set()

    for _, text in scored:

        key = norm_key(text)

        if key in seen:
            continue

        seen.add(key)
        negatives.append(text)

        if (
            len(negatives)
            >= count
        ):
            break

    return negatives


dataset_records = []

for row in verified:

    dataset_records.append({
        "sample_id":
            row["sample_id"],

        "source_file":
            row["file"],

        "sheet":
            row["sheet"],

        "doc_type_code":
            row.get(
                "doc_type_code",
                ""
            ),

        "query":
            build_query(row),

        "pos": [
            build_target_text(row)
        ],

        "neg":
            mine_negatives(row),

        # Metadata thôi, không nhúng vào text.
        "target_data_field_id":
            str(
                row[
                    "target_data_field_id"
                ]
            )
    })

print(
    "Dataset samples:",
    len(dataset_records)
)

if dataset_records:
    print(
        json.dumps(
            dataset_records[0],
            ensure_ascii=False,
            indent=2
        )
    )

# Phần 5 — Split Train / Validation / Test

Split theo **file Excel**, không split theo cột.

Ví dụ đúng:

```text
HaNoi.xlsx     → train
HaiPhong.xlsx  → train
DaNang.xlsx    → validation
HCM.xlsx       → test
```

Không làm:

```text
HaNoi cột 1,2,3 → train
HaNoi cột 4,5   → test
```

In [ ]:
# CELL 14 - Workbook Split

manifest_by_file = {
    row["file"]: row
    for row in manifest
}

files = sorted({
    row["source_file"]
    for row in dataset_records
})

train_files = set()
validation_files = set()
test_files = set()
unassigned = []

for file in files:

    split = norm_key(
        manifest_by_file
        .get(file, {})
        .get("split", "")
    )

    if split == "train":
        train_files.add(file)

    elif split in {
        "validation",
        "val"
    }:
        validation_files.add(file)

    elif split == "test":
        test_files.add(file)

    else:
        unassigned.append(file)


rng = random.Random(SEED)
rng.shuffle(unassigned)

n = len(unassigned)

if n >= 3:

    n_test = max(
        1,
        round(
            n * TEST_RATIO
        )
    )

    n_val = max(
        1,
        round(
            n * VAL_RATIO
        )
    )

    while (
        n_test + n_val
        >= n
    ):
        if n_test > 1:
            n_test -= 1
        elif n_val > 1:
            n_val -= 1
        else:
            break

    test_files.update(
        unassigned[:n_test]
    )

    validation_files.update(
        unassigned[
            n_test:
            n_test + n_val
        ]
    )

    train_files.update(
        unassigned[
            n_test + n_val:
        ]
    )

elif n == 2:

    train_files.add(
        unassigned[0]
    )

    test_files.add(
        unassigned[1]
    )

elif n == 1:

    train_files.add(
        unassigned[0]
    )


assert train_files.isdisjoint(
    validation_files
)

assert train_files.isdisjoint(
    test_files
)

assert validation_files.isdisjoint(
    test_files
)


train_data = [
    row
    for row in dataset_records
    if row["source_file"]
    in train_files
]

validation_data = [
    row
    for row in dataset_records
    if row["source_file"]
    in validation_files
]

test_source = [
    row
    for row in dataset_records
    if row["source_file"]
    in test_files
]


print("WORKBOOK SPLIT")
print("Train:", len(train_files))
print("Validation:", len(validation_files))
print("Test:", len(test_files))

print("\nSAMPLE SPLIT")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_source))

In [ ]:
# CELL 15 - Export dataset

write_jsonl(
    TRAIN_PATH,
    train_data
)

write_jsonl(
    VAL_PATH,
    validation_data
)

# Test input KHÔNG chứa đáp án.
test_data = []

ground_truth = []

for row in test_source:

    test_data.append({
        "sample_id":
            row["sample_id"],

        "source_file":
            row["source_file"],

        "sheet":
            row["sheet"],

        "doc_type_code":
            row["doc_type_code"],

        "query":
            row["query"]
    })

    ground_truth.append({
        "sample_id":
            row["sample_id"],

        "correct_data_field_id":
            row[
                "target_data_field_id"
            ]
    })

write_jsonl(
    TEST_PATH,
    test_data
)

write_jsonl(
    GROUND_TRUTH_PATH,
    ground_truth
)

print("DONE")
print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)
print(GROUND_TRUTH_PATH)

In [ ]:
# CELL 16 - Update Manifest split

new_manifest = []

for row in manifest:

    row = dict(row)

    file = row["file"]

    if file in train_files:
        row["split"] = "train"

    elif file in validation_files:
        row["split"] = "validation"

    elif file in test_files:
        row["split"] = "test"

    new_manifest.append(row)

write_jsonl(
    MANIFEST_PATH,
    new_manifest
)

print(
    "Đã update Manifest."
)

In [ ]:
# CELL 17 - Quality checks

train_ids = {
    row["sample_id"]
    for row in train_data
}

validation_ids = {
    row["sample_id"]
    for row in validation_data
}

test_ids = {
    row["sample_id"]
    for row in test_data
}

assert train_ids.isdisjoint(
    validation_ids
)

assert train_ids.isdisjoint(
    test_ids
)

assert validation_ids.isdisjoint(
    test_ids
)


# Kiểm tra cùng query nhưng bị gán DataField khác nhau.
query_to_targets = \
    defaultdict(set)

for row in dataset_records:

    query_to_targets[
        norm_key(
            row["query"]
        )
    ].add(
        row[
            "target_data_field_id"
        ]
    )

conflicts = {
    query: targets
    for query, targets
    in query_to_targets.items()
    if len(targets) > 1
}

print(
    "Conflicting labels:",
    len(conflicts)
)

if conflicts:
    for query, targets in list(
        conflicts.items()
    )[:10]:
        print(
            "\nQUERY:",
            query
        )

        print(
            "TARGETS:",
            targets
        )

assert not conflicts, (
    "Có query giống nhau nhưng "
    "ground truth khác nhau."
)

print(
    "✓ Dataset quality check passed."
)

In [ ]:
# CELL 18 - Statistics

print("=== DATASET SUMMARY ===")

print(
    "Raw workbook:",
    len(raw_files)
)

print(
    "Mapping candidate:",
    len(labels)
)

print(
    "Verified:",
    len(verified)
)

print(
    "Unique DataField:",
    len(FIELD_POOL)
)

print(
    "Train:",
    len(train_data)
)

print(
    "Validation:",
    len(validation_data)
)

print(
    "Test:",
    len(test_data)
)


doc_type_counts = Counter(
    row.get(
        "doc_type_code",
        ""
    )
    for row in dataset_records
)

print("\nDocType distribution:")

for key, value in \
    doc_type_counts.most_common():

    print(
        key or "<UNKNOWN>",
        ":",
        value
    )


if train_data:

    print("\n===== SAMPLE TRAIN =====")

    print(
        json.dumps(
            train_data[0],
            ensure_ascii=False,
            indent=2
        )
    )

# Cách chạy thực tế

## Lần đầu

### 1. Ném Excel vào

```text
Data/Raw/
```

Ví dụ:

```text
Raw/
├── Sở Tư pháp Hà Nội_01c.xlsx
├── UBND Phường Láng_01a.xlsx
├── Tổng hợp biểu mẫu_KHTC.BTP.xlsx
└── ...
```

### 2. Chạy Cell 1 → Cell 9

Sinh:

```text
tool/work/Labels/mappings.jsonl
```

### 3. Gán ground truth

Điền:

```json
{
  "target_data_field_id": "...",
  "target_field_name": "...",
  "target_field_title": "...",
  "target_column_code": "(3)",
  "verified": true,
  "verification_method": "human"
}
```

### 4. Chạy Cell 11 → Cell 18

Sinh:

```text
tool/artifacts/dataset/Train/train.jsonl
tool/artifacts/dataset/Validation/validation.jsonl
tool/artifacts/dataset/Test/test.jsonl
tool/artifacts/dataset/Test/ground_truth.jsonl
```

---

# Format dataset cuối cho BGE-M3

```json
{
  "query": "...",
  "pos": [
    "..."
  ],
  "neg": [
    "...",
    "...",
    "..."
  ]
}
```

Đây là format phù hợp với hướng:

```text
BGE-M3
+
Contrastive Learning
+
Hard Negative Mining
+
Domain Fine-tuning
```

## Điều quan trọng

Nhãn chỉ được đưa vào dataset khi `verification_method` là `human`, `reviewed` hoặc `curated`. Nhãn legacy thiếu trường này sẽ quay về hàng chờ duyệt.

`DataFieldId` **không được nhúng vào nội dung positive text**.

ID chỉ là ground truth metadata.

Model phải học:

```text
ngữ nghĩa Excel
→ ngữ nghĩa DataField
```

chứ không học thuộc:

```text
"Tổng số" → 10503
```